# 🌍 Travel Recommendation System - Jupyter Notebook

Welcome to the beginner-friendly development notebook for the Travel Recommendation System! 

In this notebook, we will develop the recommendation core step-by-step using:
1. **Pandas** for loading and exploring our dataset.
2. **TF-IDF (Term Frequency-Inverse Document Frequency)** to convert text features (descriptions, activities, types) into mathematical vectors.
3. **Cosine Similarity** to calculate how similar two destinations are.

Let's get started!

## 🛠️ Step 1: Import Libraries

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import Image, display
import os

## 📂 Step 2: Load and Preview the Data

Our dataset `travel_data_with_activities_final.csv` contains 15 popular travel destinations with columns for Destination, Country, Type, Description, Activities, and a pre-prepared `combined_features` column.

In [ ]:
# Load the CSV dataset
df = pd.read_csv("travel_data_with_activities_final.csv")

# Display the first 5 rows to verify structure
df.head()

## ⚙️ Step 3: Vectorize the Text features

Computers don't understand text descriptions directly, so we use **TF-IDF (Term Frequency-Inverse Document Frequency)**. 
- TF-IDF turns words into numbers.
- It penalizes common stop-words (like 'the', 'is', 'and') using `stop_words="english"`.
- It emphasizes unique keywords like 'surfing', 'hiking', or 'palace'.

In [ ]:
# Initialize the Vectorizer
vectorizer = TfidfVectorizer(stop_words="english")

# Fit and transform the combined_features text into a sparse matrix of numbers
tfidf_matrix = vectorizer.fit_transform(df["combined_features"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Number of unique words learned:", len(vectorizer.get_feature_names_out()))

## 📐 Step 4: Calculate Cosine Similarity

Now that each city is represented as a numerical vector, we can calculate the **Cosine Similarity** between all pairs of cities.
- Cosine Similarity measures the angle between two vectors in a multi-dimensional space.
- A score of `1.0` means they are identical, while `0.0` means they have nothing in common.

In [ ]:
# Compute cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix)

# Look at the similarity matrix shape (should be 15x15)
print("Cosine Similarity Matrix Shape:", cosine_sim.shape)

## 🧪 Step 5: Define Recommendation Functions

### Function A: Recommend similar destinations based on a place you like

In [ ]:
def recommend_by_destination(destination, top_n=3):
    # Check if the destination exists in our database
    if destination not in df["Destination"].values:
        print(f"❌ '{destination}' not found in dataset!")
        return []
    
    # Get the index of the matching destination
    idx = df[df["Destination"] == destination].index[0]
    
    # Get list of similarity scores with all other places
    scores = list(enumerate(cosine_sim[idx]))
    
    # Sort based on similarity scores (element index 1) in descending order
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    # Skip the first element because it's the requested place itself (100% similar to itself)
    scores = scores[1 : top_n + 1]
    
    # Extract matched row items
    recommendations = []
    for i in scores:
        recommendations.append(df.iloc[i[0]])
        
    return recommendations

### Function B: Recommend cities by activity

In [ ]:
def recommend_by_activity(activity, top_n=3):
    # Find rows where the 'Activities' column contains the query activity
    matches = df[df["Activities"].str.contains(activity, case=False, na=False)]
    
    # Convert matching rows to a list of dicts/records
    return matches.head(top_n).to_dict(orient="records")

## 🚀 Step 6: Test the Recommendation Functions

### Test 1: Get destinations similar to **Goa**

In [ ]:
results_goa = recommend_by_destination("Goa")

print("--- Recommendations similar to Goa ---")
for r in results_goa:
    print(f"🌟 {r['Destination']} ({r['Country']}) - Type: {r['Type']}")
    print(f"   Description: {r['Description']}")
    print(f"   Activities: {r['Activities']}\n")

### Test 2: Find destinations offering **diving**

In [ ]:
results_diving = recommend_by_activity("diving")

print("--- Recommendations with 'diving' activity ---")
for r in results_diving:
    print(f"🌊 {r['Destination']} ({r['Country']}) - Type: {r['Type']}")
    print(f"   Activities: {r['Activities']}\n")

## 🖼️ Step 7: (Optional) Display Destination Images

Let's write a small helper to load and display our generated travel images directly in the notebook!

In [ ]:
def show_destination_image(destination):
    # Try .png or .jpg
    for ext in [".png", ".jpg", ".jpeg"]:
        img_path = f"images/{destination}{ext}"
        if os.path.exists(img_path):
            print(f"Showing image for {destination}:")
            display(Image(filename=img_path, width=400))
            return
    print(f"⚠️ No image found for {destination} in images/")

# Try showing the image for Goa
show_destination_image("Goa")